# Torch model generation

                  .*****.
               .*########*.
             .################.
            .######    .######*.
           .#####   *##  .#####.
           .####   *####  .####.
            .###.   *##  .####.
             .####.    .#####.
               .*##########*.
             .##*.  .####*.
            .#####.   *##.
            .######.    *.
             .*####*.
                *##*.
                  *.

## Description

This notebook is responsible for generating the torch features from the raw data. It retrieves the data from Iceberg, applies the necessary transformations and saves the features in a format that can be used for training the model.

It trains the model using the generated features and evaluates its performance. The trained model is then saved for later use in inference.



In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def _find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("Unable to locate project root containing configs/config.yaml")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("POLARS_IMPORT_INTERVAL_AS_STRUCT", "1")

print(f"Notebook cwd: {Path.cwd()}")

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Retrieve data from Iceberg

In [ ]:
from src.repository.duckdb import BlobRepository

repository = BlobRepository()
df = repository.get_data_for_training()

df

## Feature engineering

Create the features that will be used for training the model. This includes both the features that will be used as input to the model and the target variable that we want to predict.

#### Features enabled

- Seasonality
- Harmonic features
- Business hours

In [ ]:
from src.features.v1 import FeaturesEngineeringV1

fe = FeaturesEngineeringV1()
fe.generate_torch_features(df)